In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import json

df = pd.read_csv("data/global_ads_performance_dataset.csv")
df['log_revenue'] = np.log(df['revenue'])
df['log_spend'] = np.log(df['ad_spend'])
df['month'] = pd.to_datetime(df['date']).dt.month
df['platform_clean'] = df['platform'].str.replace(' ', '_')

model_naive = smf.ols('log_revenue ~ C(platform_clean) + log_spend', data=df).fit(cov_type='HC3')

print("=== NAIVE MODEL (chưa kiểm soát confound) ===")
print(model_naive.summary().tables[1])

model_adj = smf.ols('log_revenue ~ C(platform_clean) + log_spend + C(industry) + C(country) + C(campaign_type) + C(month)', data=df).fit(cov_type='HC3')

print("\n=== ADJUSTED MODEL (đã kiểm soát confound) ===")
print(model_adj.summary().tables[1])

print("\n=== SO SÁNH PLATFORM EFFECT: NAIVE vs ADJUSTED (baseline = Google Ads) ===")
comparison = {}
for p in ['Meta_Ads', 'TikTok_Ads']:
    param_name = f'C(platform_clean)[T.{p}]'
    naive_coef = model_naive.params.get(param_name, np.nan)
    adj_coef = model_adj.params.get(param_name, np.nan)
    naive_pct = (np.exp(naive_coef) - 1) * 100
    adj_pct = (np.exp(adj_coef) - 1) * 100
    print(f"{p}: naive = {naive_pct:+.1f}% revenue vs Google | adjusted = {adj_pct:+.1f}% revenue vs Google")
    comparison[p] = {
        "naive_pct_vs_google": round(naive_pct,1),
        "adjusted_pct_vs_google": round(adj_pct,1),
        "shrinkage": round(naive_pct - adj_pct, 1)
    }

output = {
    "naive_r2": round(model_naive.rsquared,3),
    "adjusted_r2": round(model_adj.rsquared,3),
    "comparison": comparison
}
with open("dashboard_data/step6_causal_platform_effect.json", "w") as f:
    json.dump(output, f, indent=2)
print("\n✅ Saved: dashboard_data/step6_causal_platform_effect.json")

=== NAIVE MODEL (chưa kiểm soát confound) ===
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                           2.7069      0.195     13.912      0.000       2.326       3.088
C(platform_clean)[T.Meta_Ads]       0.3052      0.054      5.660      0.000       0.199       0.411
C(platform_clean)[T.TikTok_Ads]     0.7457      0.054     13.751      0.000       0.639       0.852
log_spend                           0.8094      0.022     37.164      0.000       0.767       0.852

=== ADJUSTED MODEL (đã kiểm soát confound) ===
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept                           2.7266      0.220     12.390      0.000       2.295       3.158
C(plat